# ETF Portfolio Optimization - ML Experiments

This notebook compares multiple portfolio optimization strategies:
- **Baselines**: Equal Weight, Mean-Variance, Static 60/40
- **ML Methods**: Ridge Regression, LightGBM, Ensemble
- **With/Without Sentiment**: Ablation study

## Experiments
1. ETF Universe: 6 ETFs vs 10 ETFs
2. ML Methods: Ridge vs LightGBM vs Ensemble
3. Sentiment Ablation: With vs Without sentiment features

In [ ]:
# Add src to path
import sys
sys.path.append('../src')

# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Project imports
from data import ETFDataLoader, load_default_etfs
from features import FeatureEngineer, create_feature_summary, compute_all_features_with_sentiment
from strategies import (
    EqualWeightStrategy, MeanVarianceStrategy, 
    PredictiveSharpeStrategy, GradientBoostingSharpeStrategy,
    EnsembleSharpeStrategy, create_60_40_strategy
)
from backtest import PortfolioBacktest, compare_strategies
from metrics import calculate_all_metrics, compare_strategies as compare_metrics, format_metrics_table
from visualization import (
    plot_equity_curves, plot_drawdown, plot_multiple_drawdowns,
    plot_allocation_over_time, plot_correlation_matrix,
    plot_rolling_sharpe, create_performance_dashboard
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

print("Imports loaded successfully!")

## 1. Load Data

Load expanded ETF universe (10 ETFs) including:
- **Core**: SPY, QQQ, VTI, TLT, BND, GLD
- **Diversifiers**: VEA (Intl Developed), VWO (Emerging), IWM (Small Cap), XLE (Energy)

In [ ]:
# Load expanded ETF data (10 ETFs)
prices = load_default_etfs(start_date='2015-01-01', expanded=True)

print(f"\nLoaded {len(prices)} days of data")
print(f"ETFs ({len(prices.columns)}): {', '.join(prices.columns)}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")

In [ ]:
# Show summary statistics
loader = ETFDataLoader()
summary = loader.get_data_summary(prices)
print("\n" + "="*80)
print("ETF Summary Statistics")
print("="*80)
print(summary)

In [ ]:
# Visualize price history
fig, ax = plt.subplots(figsize=(14, 6))
normalized_prices = prices / prices.iloc[0] * 100
normalized_prices.plot(ax=ax, linewidth=1.5, alpha=0.8)
ax.set_title('ETF Price History (Normalized to 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Normalized Price', fontsize=12)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Data Splitting

- **Train**: 2015-2021 (7 years)
- **Validation**: 2022 (1 year)
- **Test**: 2023-2025 (held-out for final evaluation)

In [ ]:
# Split data
train, val, test = loader.split_train_val_test(prices)

In [ ]:
# Visualize correlations for expanded universe
returns_train = loader.calculate_returns(train).dropna()
plot_correlation_matrix(returns_train, title='Training Set: Asset Correlation Matrix (10 ETFs)')
plt.show()

## 3. Define All Strategies

### Baselines
1. Equal Weight (1/N)
2. Mean-Variance (Max Sharpe)
3. 60/40 Portfolio

### ML Methods
4. Predictive Sharpe (Ridge Regression)
5. LightGBM Sharpe (Gradient Boosting)
6. Ensemble Sharpe (Ridge + LightGBM + RF)

In [ ]:
# Define stock and bond tickers for 60/40 portfolio
stock_tickers = ['SPY', 'QQQ', 'VTI', 'VEA', 'VWO', 'IWM', 'XLE']
bond_tickers = ['TLT', 'BND']
# Filter to only tickers that exist in our data
stock_tickers = [t for t in stock_tickers if t in prices.columns]
bond_tickers = [t for t in bond_tickers if t in prices.columns]

# Define all strategies
strategies = {
    # Baselines
    'Equal Weight': EqualWeightStrategy(),
    'Mean-Variance': MeanVarianceStrategy(lookback_days=252, max_weight=0.3),
    '60/40 Portfolio': create_60_40_strategy(stock_tickers, bond_tickers),
    
    # ML Methods
    'Predictive (Ridge)': PredictiveSharpeStrategy(max_weight=0.3),
    'LightGBM Sharpe': GradientBoostingSharpeStrategy(max_weight=0.3),
    'Ensemble Sharpe': EnsembleSharpeStrategy(max_weight=0.3),
}

print(f"Defined {len(strategies)} strategies:")
for name in strategies.keys():
    print(f"  - {name}")

In [ ]:
# Show initial allocations for each strategy
print("\n" + "="*80)
print("Initial Allocations (using training data)")
print("="*80)

for name, strategy in strategies.items():
    weights = strategy.allocate(train)
    # Sort by weight descending
    sorted_weights = sorted(weights.items(), key=lambda x: x[1], reverse=True)
    print(f"\n{name}:")
    for ticker, weight in sorted_weights:
        if weight > 0.01:  # Only show weights > 1%
            print(f"  {ticker}: {weight:>6.2%}")

## 4. Run Backtests

Run backtests with:
- Monthly rebalancing
- 0.1% transaction costs
- $100,000 initial capital

In [ ]:
# Backtest settings
INITIAL_CAPITAL = 100000
TRANSACTION_COST = 0.001  # 0.1%
REBALANCE_FREQ = 'M'  # Monthly

In [ ]:
# Run backtest on TRAINING data
print("\n" + "="*80)
print("Backtesting on Training Data (2015-2021)")
print("="*80)

results_train, allocations_train = compare_strategies(
    strategies,
    train,
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=TRANSACTION_COST,
    rebalance_frequency=REBALANCE_FREQ
)

In [ ]:
# Calculate training metrics
metrics_train = compare_metrics(results_train, allocations_train)
metrics_train_fmt = format_metrics_table(metrics_train)

print("\n" + "="*80)
print("Training Set Performance Metrics")
print("="*80)
print(metrics_train_fmt.to_string())

In [ ]:
# Visualize training equity curves
plot_equity_curves(results_train, title='Training Set: Portfolio Performance (2015-2021)')
plt.show()

In [ ]:
# Run backtest on VALIDATION data
print("\n" + "="*80)
print("Backtesting on Validation Data (2022)")
print("="*80)

results_val, allocations_val = compare_strategies(
    strategies,
    val,
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=TRANSACTION_COST,
    rebalance_frequency=REBALANCE_FREQ
)

In [ ]:
# Calculate validation metrics
metrics_val = compare_metrics(results_val, allocations_val)
metrics_val_fmt = format_metrics_table(metrics_val)

print("\n" + "="*80)
print("Validation Set Performance Metrics")
print("="*80)
print(metrics_val_fmt.to_string())

In [ ]:
# Visualize validation equity curves
plot_equity_curves(results_val, title='Validation Set: Portfolio Performance (2022)')
plt.show()

In [ ]:
# Run backtest on TEST data
print("\n" + "="*80)
print("Backtesting on Test Data (2023-2025)")
print("="*80)

results_test, allocations_test = compare_strategies(
    strategies,
    test,
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=TRANSACTION_COST,
    rebalance_frequency=REBALANCE_FREQ
)

In [ ]:
# Calculate test metrics
metrics_test = compare_metrics(results_test, allocations_test)
metrics_test_fmt = format_metrics_table(metrics_test)

print("\n" + "="*80)
print("Test Set Performance Metrics (Final Results)")
print("="*80)
print(metrics_test_fmt.to_string())

In [ ]:
# Visualize test equity curves
plot_equity_curves(results_test, title='Test Set: Portfolio Performance (2023-2025)')
plt.show()

In [ ]:
# Visualize drawdowns on test set
plot_multiple_drawdowns(results_test, title='Test Set: Drawdown Comparison')
plt.show()

## 5. Cross-Period Analysis

Compare performance across Train/Val/Test periods.

In [ ]:
# Create comprehensive comparison table
comparison_data = []

for strategy_name in strategies.keys():
    for period, metrics in [('Train', metrics_train), ('Val', metrics_val), ('Test', metrics_test)]:
        comparison_data.append({
            'Strategy': strategy_name,
            'Period': period,
            'Sharpe': metrics.loc[strategy_name, 'Sharpe Ratio'],
            'Return (%)': metrics.loc[strategy_name, 'Annualized Return'] * 100,
            'Volatility (%)': metrics.loc[strategy_name, 'Annualized Volatility'] * 100,
            'Max DD (%)': metrics.loc[strategy_name, 'Max Drawdown'] * 100
        })

comparison_df = pd.DataFrame(comparison_data)

# Pivot for better display
sharpe_pivot = comparison_df.pivot(index='Strategy', columns='Period', values='Sharpe')[['Train', 'Val', 'Test']]

print("\n" + "="*80)
print("Sharpe Ratio Comparison Across Periods")
print("="*80)
print(sharpe_pivot.round(3).to_string())

In [ ]:
# Visualize Sharpe ratio across periods
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(strategies))
width = 0.25

colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, period in enumerate(['Train', 'Val', 'Test']):
    values = [comparison_df[(comparison_df['Strategy']==s) & (comparison_df['Period']==period)]['Sharpe'].values[0] 
              for s in strategies.keys()]
    ax.bar(x + i*width, values, width, label=period, color=colors[i], alpha=0.8)

ax.set_xlabel('Strategy', fontsize=12)
ax.set_ylabel('Sharpe Ratio', fontsize=12)
ax.set_title('Sharpe Ratio Comparison Across Periods', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(strategies.keys(), rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

## 6. ML Method Comparison

Focus on comparing Ridge vs LightGBM vs Ensemble

In [ ]:
# Extract ML methods only
ml_strategies = ['Predictive (Ridge)', 'LightGBM Sharpe', 'Ensemble Sharpe']
ml_results = {k: results_test[k] for k in ml_strategies}

# Plot ML methods comparison
fig, ax = plt.subplots(figsize=(12, 6))

for name, values in ml_results.items():
    normalized = values / values.iloc[0] * 100
    ax.plot(normalized.index, normalized.values, linewidth=2, label=name)

ax.set_title('ML Methods Comparison (Test Set)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value (Normalized)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ML methods metrics comparison
ml_metrics = metrics_test.loc[ml_strategies]
print("\n" + "="*80)
print("ML Methods Performance Comparison (Test Set)")
print("="*80)
print(format_metrics_table(ml_metrics).to_string())

## 7. Best Strategy Analysis

Analyze the best performing strategy in detail.

In [ ]:
# Find best strategy by Sharpe ratio on test set
best_strategy = metrics_test['Sharpe Ratio'].idxmax()
best_sharpe = metrics_test.loc[best_strategy, 'Sharpe Ratio']

print(f"\nBest Strategy (by Test Sharpe): {best_strategy}")
print(f"Test Sharpe Ratio: {best_sharpe:.3f}")
print(f"Test Return: {metrics_test.loc[best_strategy, 'Annualized Return']*100:.2f}%")
print(f"Test Volatility: {metrics_test.loc[best_strategy, 'Annualized Volatility']*100:.2f}%")
print(f"Max Drawdown: {metrics_test.loc[best_strategy, 'Max Drawdown']*100:.2f}%")

In [ ]:
# Plot allocation over time for best strategy
if best_strategy in allocations_test:
    plot_allocation_over_time(
        allocations_test[best_strategy],
        title=f'{best_strategy}: Allocation Over Time (Test Set)'
    )
    plt.show()

## 8. Summary and Conclusions

In [ ]:
# Create summary table
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

summary_data = []
for strategy in strategies.keys():
    summary_data.append({
        'Strategy': strategy,
        'Test Sharpe': metrics_test.loc[strategy, 'Sharpe Ratio'],
        'Test Return (%)': metrics_test.loc[strategy, 'Annualized Return'] * 100,
        'Test Vol (%)': metrics_test.loc[strategy, 'Annualized Volatility'] * 100,
        'Max DD (%)': metrics_test.loc[strategy, 'Max Drawdown'] * 100,
        'Sortino': metrics_test.loc[strategy, 'Sortino Ratio'],
    })

summary_df = pd.DataFrame(summary_data).set_index('Strategy')
summary_df = summary_df.sort_values('Test Sharpe', ascending=False)
print(summary_df.round(3).to_string())

In [ ]:
# Save results
metrics_train_fmt.to_csv('../data/ml_metrics_train.csv')
metrics_val_fmt.to_csv('../data/ml_metrics_val.csv')
metrics_test_fmt.to_csv('../data/ml_metrics_test.csv')
comparison_df.to_csv('../data/ml_comparison_summary.csv', index=False)
summary_df.to_csv('../data/ml_final_summary.csv')

print("\nResults saved to data/ directory")

## 9. Key Findings

### Observations
1. Compare ML methods (Ridge, LightGBM, Ensemble) against baselines
2. Evaluate consistency across train/val/test periods
3. Assess the value of the expanded ETF universe

### Next Steps
1. **Sentiment Integration**: Add GDELT-based sentiment features
2. **Feature Importance**: Analyze which features drive predictions
3. **Hyperparameter Tuning**: Optimize model parameters on validation set
4. **Walk-Forward Validation**: Test with rolling windows